In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length

# Spark oturumunu başlatma
spark = SparkSession.builder \
    .appName('TwitterAnalysis') \
    .getOrCreate()

print("Spark Session Başlatıldı!")

dosya_yolu = '/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv'
# PySpark'ın metinleri kaydırmadan doğru okuması için quote, escape ve multiLine parametrelerini ekliyoruz
spark_df = spark.read.csv(
    dosya_yolu,
    header=True,
    inferSchema=True,
    quote='"',       # Tırnak içindeki metinleri tek parça algıla
    escape='"',      # Tırnak içindeki virgülleri ayraç sanma
    multiLine=True   # Alt satıra inen tweetleri destekle
)

# Verinin şemasını (schema) yazdıralım
spark_df.printSchema()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
print("--- 1. Dönüşüm (Transformation) İşlemleri ---")

# İşlem 1: select() ve filter() kullanımı
# Sadece "Delta" havayoluna ait pozitif tweetlerin metinlerini ve güven skorlarını seçelim
print("\n1. Delta Havayollarının Pozitif Tweetleri (select & filter):")
delta_pozitif = spark_df.filter((col("airline") == "Delta") & (col("airline_sentiment") == "positive")) \
                        .select("airline", "text", "airline_sentiment_confidence")
delta_pozitif.show(3, truncate=False)

# İşlem 2: withColumn() kullanımı
# Tweet metninin karakter uzunluğunu hesaplayıp yeni bir sütun olarak ekleyelim
print("\n2. Tweet Uzunluğu Sütunu Ekleme (withColumn):")
df_with_length = spark_df.withColumn("tweet_uzunlugu", length(col("text")))
df_with_length.select("text", "tweet_uzunlugu").show(3, truncate=True)

# İşlem 3: groupBy() kullanımı
# Havayolu şirketlerine göre tweet sayılarını gruplayalım
print("\n3. Havayolu Şirketlerine Göre Tweet Sayıları (groupBy):")
sirket_gruplari = spark_df.groupBy("airline").count()
sirket_gruplari.show()

--- 1. Dönüşüm (Transformation) İşlemleri ---

1. Delta Havayollarının Pozitif Tweetleri (select & filter):


NameError: name 'spark_df' is not defined

In [ ]:
print("--- 2. Eylem (Action) İşlemleri ---")

# İşlem 4: count() kullanımı
toplam_kayit = spark_df.count()
print(f"Veri setindeki toplam kayıt sayısı: {toplam_kayit}\n")

# İşlem 5: describe() kullanımı
# Sayısal bir sütun olan "airline_sentiment_confidence" (güven skoru) için temel istatistikler
print("Güven Skoru (Confidence) İstatistiksel Özeti (describe):")
spark_df.describe(['airline_sentiment_confidence']).show()

In [ ]:
print("--- 3. Spark SQL İşlemleri ---")

# Geçici görünüm (TempView) oluşturma
spark_df.createOrReplaceTempView('tweets_table')

# İşlem 6: Spark SQL Sorgusu 1
# Negatif tweetlerin en çok hangi sebeplerden (negativereason) kaynaklandığını bulma
print("\nSorgu 1: En Sık Rastlanan Negatif Tweet Sebepleri:")
sql_sorgu1 = spark.sql("""
    SELECT negativereason, COUNT(*) as sebep_sayisi
    FROM tweets_table
    WHERE airline_sentiment = 'negative' AND negativereason != 'Belirtilmedi'
    GROUP BY negativereason
    ORDER BY sebep_sayisi DESC
    LIMIT 5
""")
sql_sorgu1.show()

# İşlem 7: Spark SQL Sorgusu 2
# Duygu durumlarına (sentiment) göre ortalama güven skorlarını hesaplama
print("\nSorgu 2: Duygu Durumuna Göre Ortalama Güven Skoru:")
sql_sorgu2 = spark.sql("""
    SELECT airline_sentiment, AVG(airline_sentiment_confidence) as ortalama_guven
    FROM tweets_table
    GROUP BY airline_sentiment
""")
sql_sorgu2.show()

In [ ]:
import pandas as pd
import time

print("--- 4. Pandas ve PySpark Performans Karşılaştırması ---")

# Aynı veriyi Pandas ile okuyalım
pandas_df = pd.read_csv('/content/drive/MyDrive/Buyuk_Veri_Donem_Projesi/twitter_big_data_pipeline/data/processed/cleaned_tweets.csv')

# --- PANDAS TESTİ ---
baslangic_pandas = time.time()
# Pandas ile şirket bazlı tweet sayma işlemi
pandas_sonuc = pandas_df.groupby('airline').size()
bitis_pandas = time.time()
pandas_sure = bitis_pandas - baslangic_pandas

# --- PYSPARK TESTİ ---
baslangic_spark = time.time()
# PySpark ile şirket bazlı tweet sayma işlemi (.collect() ile action tetikliyoruz)
spark_sonuc = spark_df.groupBy("airline").count().collect()
bitis_spark = time.time()
spark_sure = bitis_spark - baslangic_spark

print(f"Pandas Çalışma Süresi:  {pandas_sure:.5f} saniye")
print(f"PySpark Çalışma Süresi: {spark_sure:.5f} saniye")

print("\n(Not: Veri seti nispeten küçük (~14.000 satır) olduğu için Pandas bu seviyede "
      "\nPySpark'tan daha hızlı veya benzer sürede çalışabilir. PySpark'ın gerçek gücü "
      "\nmilyonlarca satırlık verilerde dağıtık işleme yaparken ortaya çıkmaktadır.)")

spark.stop()